This notebook is responsible for:
- Training and validating the GBM model across progressively wider datasets.
- Exporting the model parameters for testing evaluation in another notebook.

# Library Dependencies

In [1]:
#!pip install -qq joblib ipython numpy pandas scikit-learn xgboost

In [2]:
import joblib, json, sys, warnings
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import sklearn
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import xgboost
from xgboost import XGBRegressor


In [3]:
!python --version
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)

Python 3.10.18
numpy 2.2.6
pandas 2.3.3
scikit-learn 1.7.2


# Feature Set Specification, Data Loading, and Training/Validation Preparation

In [4]:
# AUTOREGRESSIVE (LAG) FEATURES

# lags = sorted(set(
#     list(range(1, 4)) + [6, 12, 18, 24, 36, 48, 72, 168]
#     + list(range(24, 27)) + [36, 48]
#     + [24*i for i in range(3,7)]
#     + [24*7*i for i in range(1,5)]
# ))

lags = [1, 2, 3, 6, 12, 18, 24, 36, 48, 72, 168]
lagFeatures = [f"Adjusted demand -{h} hr" for h in lags]

In [5]:
# CALENDAR FEATURES

# Raw integer calendar features
intDateTimeFeatures = ["Hour", "Month", "DayOfWeek", "DayOfYear"]

# Low order hour of day and day of year Fourier term features
hourFourierFeatures, dayFourierFeatures = [], []
for i in (1, 2, 3):
    argStr = (f"{i}*" if i>1 else "") + "Hour"
    hourFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])
    argStr = (f"{i}*" if i>1 else "") + "DayOfYear"
    dayFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])

# Hour of day and day of week one-hot encodings.
# Weekend and holiday flags.
hourDummyFeatures = [f"Hour_Flag_{h}" for h in range(24)]
dayDummyFeatures = ["Day_Flag_Weekend", "Day_Flag_Holiday"]
dayDummyFeatures += [f"DayOfWeek_Flag_{d}" for d in range(7)]
monthDummyFeatures = [f"Month_Flag_{m}" for m in range(1, 13)]

# Collate all calendar features
calendarFeatures = (
    hourFourierFeatures
    + dayFourierFeatures
    + dayDummyFeatures[:2]
)


In [6]:
# ENERGY FEATURES

energyFeatures = [
    # "Adjusted net generation",
    # "Adjusted total interchange",
    "FPC", "FMPP", "SOCO", "TEC",
    "JEA", "SEC", "HST", "GVL",
]


In [7]:
# WEATHER FEATURES

skyCodes = ['BKN', 'CLR', 'FEW', 'SCT', 'OVC', 'NA']
directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW", "VRB"]

weatherFeatures = ([
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlySeaLevelPressure",
    "HourlyVisibility",
    "HourlyWindSpeed"]
    + [f"HourlySkyConditions_Flag_{code}" for code in skyCodes]
    + ["HourlyWindDirection_Flag_VRB",]
    + ["sin(HourlyWindDirection)", "cos(HourlyWindDirection)"]
)


In [8]:
# All features collated.
# Selected reduced feature set is specified, as determined by feature clustering and importance.

allFeatures = (
    lagFeatures 
    + calendarFeatures 
    + energyFeatures 
    + weatherFeatures
)
selectedFeatures = ['Day_Flag_Weekend', 'Day_Flag_Holiday', 'sin(DayOfYear)', 'HourlySeaLevelPressure', 'HourlyWindSpeed', 'sin(HourlyWindDirection)', 'cos(HourlyWindDirection)', 'Adjusted demand -6 hr', 'HourlyRelativeHumidity', 'Adjusted demand -12 hr', 'Adjusted demand -1 hr', 'SEC', 'GVL', 'cos(DayOfYear)', 'FMPP', 'TEC', 'JEA', 'cos(2*DayOfYear)', 'HourlyPrecipitation', 'HourlySkyConditions_Flag_OVC', 'HourlyVisibility', 'HourlySkyConditions_Flag_SCT', 'HourlySkyConditions_Flag_BKN', 'HourlySkyConditions_Flag_NA', 'HourlySkyConditions_Flag_CLR', 'HourlyWindDirection_Flag_VRB', 'sin(2*DayOfYear)', 'sin(3*DayOfYear)', 'cos(3*DayOfYear)']

In [9]:
# Load training and validation data.
# Prepare to store model specifications and parameters for exporting.

data_root   = Path("data/clean")
train_dir   = data_root / "train"
val_dir     = data_root / "val"

DFtrain = pd.read_pickle(train_dir / "DFtrain.pkl")
DFval   = pd.read_pickle(val_dir   / "DFval.pkl")

TARGET_COL = "Adjusted demand"
TIME_COL = "t"

MODEL_SPECS = {
    "model_A": {
        "feature_cols": lagFeatures,   # autoregressive only
    },
    "model_B": {
        "feature_cols": lagFeatures+calendarFeatures,   # + calendar features
    },
    "model_C": {
        "feature_cols": lagFeatures+calendarFeatures+energyFeatures,   # + energy features
    },
    "model_D": {
        "feature_cols": allFeatures,   # full feature set
    },
    "model_E": {
        "feature_cols": selectedFeatures,   # reduced feature set
    },
    "model_F": {
        "feature_cols": allFeatures,   # full feature set on weighted RMSE loss and evaluation
    },
}
xgb_results = {}

xgb_model_root = Path("models") / "XGB"
xgb_model_root.mkdir(parents=True, exist_ok=True)


# Model Training, Tuning, Validation, and Exporting

In [10]:
# Reproducibility
np.random.seed(42)

# Hyperparameter grid search
max_depth_grid     = [3, 5, 7]
n_estimators_grid  = [200, 400]
learning_rate_grid = [0.05, 0.1]

# Establish top quartile observations for weighted RMSE training and evaluation.
q75 = DFtrain[TARGET_COL].quantile(0.75)
print(f"75th percentile training threshold for {TARGET_COL}: {q75:.3f}")
peak_weight = 3.0

# boolean masks for peak hours (train/val) using the *training* threshold
peak_mask_train = DFtrain[TARGET_COL] >= q75
peak_mask_val   = DFval[TARGET_COL]   >= q75

# sample weights: peak hours get peak_weight, others weight 1
sample_weight_train_F = np.where(peak_mask_train.to_numpy(), peak_weight, 1.0)
sample_weight_val_F   = np.where(peak_mask_val.to_numpy(),   peak_weight, 1.0)

def eval_xgb(
    params,
    X_train,
    y_train,
    X_val,
    y_val,
    sample_weight_train=None,
    sample_weight_val=None,
):
    """
    Fit XGBRegressor with given params and return validation RMSE + model.

    If sample_weight_* is provided, uses it for training and for weighted val RMSE.
    """
    
    model = XGBRegressor(
        objective="reg:squarederror",
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=0,
        **params,
    )

    fit_kwargs = {}
    if sample_weight_train is not None:
        fit_kwargs["sample_weight"] = sample_weight_train

    model.fit(X_train, y_train, **fit_kwargs)

    y_val_pred = model.predict(X_val)

    if sample_weight_val is not None:
        rmse = root_mean_squared_error(
            y_val,
            y_val_pred,
            sample_weight=sample_weight_val,
        )
    else:
        rmse = root_mean_squared_error(y_val, y_val_pred)

    return rmse, model

# -----------------------------
# MAIN TRAINING AND TUNING LOOP
# -----------------------------

for model_name, spec in MODEL_SPECS.items():
    feature_cols = spec["feature_cols"]

    X_train = DFtrain[feature_cols].to_numpy()
    y_train = DFtrain[TARGET_COL].to_numpy()
    X_val   = DFval[feature_cols].to_numpy()
    y_val   = DFval[TARGET_COL].to_numpy()

    print(f"\n=== Tuning XGB for {model_name} ===")
    print("n_features:", len(feature_cols))

    if model_name == "model_F":
        sw_train = sample_weight_train_F
        sw_val   = sample_weight_val_F
        print("Using peak-weighted training and validation loss for XGB model_F.")
    else:
        sw_train = None
        sw_val   = None

    best_params = None
    best_rmse   = np.inf
    best_model  = None

    for max_depth in max_depth_grid:
        for n_estimators in n_estimators_grid:
            for learning_rate in learning_rate_grid:
                params = dict(
                    max_depth=max_depth,
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                )
                rmse, model = eval_xgb(
                    params,
                    X_train, y_train,
                    X_val,   y_val,
                    sample_weight_train=sw_train,
                    sample_weight_val=sw_val,
                )
                print(
                    f"{model_name}  depth={max_depth}, trees={n_estimators}, "
                    f"lr={learning_rate:.2f}  --- val RMSE: {rmse:.4f}"
                )
                if rmse < best_rmse:
                    best_rmse   = rmse
                    best_params = params
                    best_model  = model

    print(f"Best XGB for {model_name}: params={best_params}  (val RMSE={best_rmse:.4f})")

    xgb_results[model_name] = {
        "best_params": best_params,
        "best_rmse":   best_rmse,
        "feature_cols": list(feature_cols),
    }

    model_path = xgb_model_root / f"{model_name}_XGB_best.joblib"
    joblib.dump(best_model, model_path)

    meta = {
        "model_name": model_name,
        "best_params": {
            k: float(v) if isinstance(v, np.floating) else v
            for k, v in best_params.items()
        },
        "best_val_rmse": float(best_rmse),
        "feature_cols": list(feature_cols),
        "target_col": TARGET_COL,
    }

    if model_name == "model_F":
        meta["peak_weighting"] = {
            "quantile": 0.75,
            "threshold": float(q75),
            "peak_weight": float(peak_weight),
        }

    meta_path = xgb_model_root / f"{model_name}_XGB_meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Saved best {model_name} XGB model to:  {model_path}")
    print(f"Saved {model_name} XGB metadata to:    {meta_path}")

print("\nThe best validated XGB model for each of the six categories has been saved.")

75th percentile training threshold for Adjusted demand: 17395.000

=== Tuning XGB for model_A ===
n_features: 11
model_A  depth=3, trees=200, lr=0.05  --- val RMSE: 567.7756
model_A  depth=3, trees=200, lr=0.10  --- val RMSE: 520.1150
model_A  depth=3, trees=400, lr=0.05  --- val RMSE: 510.6762
model_A  depth=3, trees=400, lr=0.10  --- val RMSE: 485.6240
model_A  depth=5, trees=200, lr=0.05  --- val RMSE: 494.8166
model_A  depth=5, trees=200, lr=0.10  --- val RMSE: 472.0402
model_A  depth=5, trees=400, lr=0.05  --- val RMSE: 467.4955
model_A  depth=5, trees=400, lr=0.10  --- val RMSE: 454.0643
model_A  depth=7, trees=200, lr=0.05  --- val RMSE: 479.4411
model_A  depth=7, trees=200, lr=0.10  --- val RMSE: 472.0341
model_A  depth=7, trees=400, lr=0.05  --- val RMSE: 463.9540
model_A  depth=7, trees=400, lr=0.10  --- val RMSE: 464.2226
Best XGB for model_A: params={'max_depth': 5, 'n_estimators': 400, 'learning_rate': 0.1}  (val RMSE=454.0643)
Saved best model_A XGB model to:  models/XGB/